## Motl basics

* Motl stands for "motive list" and contains list of particles and their properties. It can be used in subtomogram averaging or to perform contextual analysis.
* Internally, the particle list is stored within a Motl class as pandas DataFrame with 20 columns (see below) and N rows, where N corresponds to number of particles. 
* Externally, the particle list can be loaded from and written as a binary file in any of the following: EM format (novaSTA, TOM/AV3, ArtiaX compatible), a RELION starfile (currently up to version 4.x), a STOPGAP starfile, IMOD .mod file, and a simple CSV file. 
* The module cryomotl contains stand stand-alone functions for hassle-free conversions between formats, as well as classes as described below

### Motl class

Motl class is a parent class containing functions that are general to all formats. It is the class that is used for all manipulations applied to the motl list. <br>
The particle list itself is stored in the member variable `df` as pandas DataFrame that has following columns:

1.  "score" - a quality metric (typically cross-correlation value between the particle and the reference)
2.  "geom1" - a free geometric property
3.  "geom2" - a free geometric property
4.  "subtomo_id" - a subtomogram id; **IMPORTANT** many functions rely on this one to be unique
5.  "tomo_id" - a tomogram id to which the particle is affiliated to
6.  "object_id" - an object id to which the particle is affiliated to
7.  "subtomo_mean" - a mean value of the subtomogram
8.  "x" - a position in the tomogram (an integer value), typically used for subtomogram extraction
9.  "y" - a position in the tomogram (an integer value), typically used for subtomogram extraction
10. "z" - a position in the tomogram (an integer value), typically used for subtomogram extraction
11. "shift_x" - shift of the particle in X direction (a float value); to complete position of a particle is given by x + shift_x
12. "shift_y" - shift of the particle in Y direction (a float value); to complete position of a particle is given by y + shift_y
13. "shift_z" - shift of the particle in Z direction (a float value); to complete position of a particle is given by z + shift_z
14. "geom3" - a free geometric property
15. "geom4" - a free geometric property
16. "geom5" - a free geometric property
17. "phi" - a phi angle describing rotation around the first Z axis (following Euler zxz convention)
18. "psi" - a psi angle describing rotation around the second Z axis (following Euler zxz convention)
19. "theta" - a theta angle describing rotation around the X axis (following Euler zxz convention)
20. "class" - a class of the particle

#### Child classes
Different subclasses include EmMotl, RelionMotl, StopgapMotl, DynamoMotl, ModMotl and contain functionalities allowing for smooth transitions between the different conventions utilised in a given software. They mostly contain funtions for reading in/out lists and are used under-the-hood to ensure compatibility with the parent Motl file, which we use in order to use functions to modify and inspect our files.


## Working with particle lists: Basic examples
In the following section, some examples of how to use Motl files in your analysis pipelines are provided. For a complete list of functions, please refer to the `cryomotl` module in the API guide. <br>
Here we provide examples for working in a Jupyter notebook. Please note that the commands that a user most often needs can be accessed directly from the terminal via the CLI (see below).

NOTE: For all the functions displayed, it is assumed that the `cryomotl` module is imported

In [4]:
import os
import pandas as pd
from cryocat.core.cryomotl import *

In [6]:
# set the path to the folder containing the test material for the examples
test_folder = "../../../tests/test_data"
os.chdir(test_folder)

### Read a particle list into a Motl object
The first step to work with a particle list is to load it and store it as a Motl object. This can be accomplished with the `load()` function, which is used to initialize the Motl class. <br>

**Example:**

In [8]:
my_motl = Motl.load('au_1.em')

Then, the properties of the particles can be displayed by inspecting the `df` attribute (pandas DataFrame).

In [9]:
display(my_motl.df)

,score,geom1,geom2,subtomo_id,tomo_id,object_id,subtomo_mean,x,y,z,shift_x,shift_y,shift_z,geom3,geom4,geom5,phi,psi,theta,class
0,0.063319,1.0,1.0,1.0,160.0,1.0,160.0,230.0,201.0,178.0,-0.169803,-0.106735,0.370416,0.0,0.0,1.0,49.184898,52.307846,128.014435,1.0
1,0.063319,1.0,2.0,2.0,160.0,1.0,160.0,222.0,185.0,183.0,-0.018877,-0.079317,-0.081875,0.0,0.0,1.0,94.184898,52.307846,128.014435,1.0
2,0.063319,1.0,3.0,3.0,160.0,1.0,160.0,208.0,175.0,176.0,-0.073773,0.039675,0.419004,0.0,0.0,1.0,139.184906,52.307846,128.014435,1.0
3,0.063319,1.0,4.0,4.0,160.0,1.0,160.0,196.0,177.0,163.0,-0.101322,0.038400,-0.319851,0.0,0.0,1.0,184.184906,52.307846,128.014435,1.0
4,0.063319,1.0,5.0,5.0,160.0,1.0,160.0,193.0,190.0,150.0,-0.055950,-0.253967,-0.250405,0.0,0.0,1.0,229.184906,52.307846,128.014435,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,0.059292,2.0,4.0,60.0,160.0,8.0,160.0,506.0,580.0,318.0,0.228478,-0.227915,0.244360,0.0,0.0,8.0,250.057617,55.644222,19.905619,1.0
60,0.059292,2.0,5.0,61.0,160.0,8.0,160.0,516.0,595.0,319.0,-0.052046,0.356272,-0.476559,0.0,0.0,8.0,295.057617,55.644222,19.905619,1.0
61,0.059292,2.0,6.0,62.0,160.0,8.0,160.0,512.0,613.0,323.0,0.421434,-0.217416,0.138664,0.0,0.0,8.0,340.057617,55.644222,19.905619,1.0
62,0.059292,2.0,7.0,63.0,160.0,8.0,160.0,498.0,622.0,329.0,-0.285294,-0.157075,0.386493,0.0,0.0,8.0,385.057617,55.644222,19.905619,1.0


This displays the content of the pandas Dataframe (`.df` attribute) with all the columns described in the previous section.

### Extract a subset of particles
To work on a subset of particles based on the value of a particular feature, you can extract them with the `get_motl_subset()` function. By default, the feature that is taken into account is the tomogram ID (`tomo_id`) and it returns a new Motl object, however you can ask for a pandas DataFrame as well. Here are some examples:

In [ ]:
subset_motl_tomo = my_motl.get_motl_subset(160) #select particles belonging to tomogram 160 and return a new Motl object
subset_motl_class = my_motl.get_motl_subset(1, column_name="class") #select particles belonging to class 1 and return a new Motl object
subset_df_tomo = my_motl.get_motl_subset(160, return_df=True) #select particles belonging to tomogram 160 and return a pandas.DataFrame

### Clean a particle list
Different functions are available to clean particle list depending on the analysis pipeline. <br/>

* **Clean particles by distance:** Clean duplicate particles, commonly required when working with particles from oversampling. The function that accomplishes this is `clean_by_distance()`. This functions changes the `df` attribute of the Motl. To save the particle list after cleaning, it is necessary to write the function to file with the `write_out()` function (see below).

In [ ]:
my_motl.clean_by_distance(10) #remove particles that are closer than 10 voxels to each other, keeping the one with the highest score value
my_motl.clean_by_distance(10, column_name="class") #remove particles that are closer than 10 voxels to each other, grouping the particles by class and keeping the one with the highest score value

* **Clean particles by CC value:** This is accomplished with the `clean_by_otsu()` function.

In [ ]:
my_motl.clean_by_otsu(column_name="tomo_id") #group the particle by tomogram and compute the Otsu's threshold to select the score threshold according to which the particles are cleaned

* **Clean particles by radius**: In case of particles the coordinates whereof were sampled on a spherical or ellipsoidal surface, the `clean_by_radius()` function allows to remove those particles that are not fitting the surface but are rather outside its radius. This function belongs to the `ParametricAssembly` class of the `assembly` module: first fit one ellipsoid per object with `ParametricAssembly.from_motl()`, then clean the particles against the fitted surfaces. If an output path is passed, the cleaned particle list will be written to the specified file.

In [ ]:
from cryocat.analysis import assembly
surfaces = assembly.ParametricAssembly.from_motl(my_motl, column_name='object_id') #fit one ellipsoid per object (grouped by tomo_id and object_id)
my_motl_cleaned = surfaces.clean_by_radius(my_motl, output_path='./my_motl_cleaned.em') #for each object, remove the particles that are outside its radius ± the standard deviation of the distance between the particles and the object center
my_motl_cleaned = surfaces.clean_by_radius(my_motl, threshold=5, output_path='./my_motl_cleaned.em') #for each object, remove the particles that are outside its radius ± the threshold value

### Write a Motl to file
If you have edited the `df` attribute of your Motl object or you generated a new Motl object and you wish to save them to a particle list file, you need to use the `write_out()` function:

In [ ]:
my_motl.write_out("path/to/desired_output_file.em") #this will save my_motl to desired_output_file.em

### Example of different classes in use
Let's assume we are working with a Stopgap motl file and want to scale the coordinates 2x in order to decrease the binning factor we are working at twice. In order to do that, we need to ensure loading the file within the conventions of the appropriate instance by specifying the type of the motl we are dealing with.

In [ ]:
my_sg_motl = Motl.load('input_motl_file.star', motl_type='stopgap')

Still using the Motl class, we modify the object according to our needs.

In [ ]:
my_sg_motl.scale_coordinates(2)
my_sg_motl.update_coordinates()

If one displays the `df` of the object at this stage, they would notice that column labels are as described above, i.e. characteristic to a Motl class object. To write out the file in the correct format for further use in stopgap, we should call:

In [ ]:
Motl.write_out(my_sg_motl,'output_path.star','stopgap') #specify motl type to ensure correct convention

We can inspect written out data by using either of the classes: Motl or Stopgap. 

In [ ]:
display(StopgapMotl.read_in('output_motl_file.star')) # keeps the specifiers (labels) and the column order of the file as is; object stored as DataFrame
display(Motl.load('output_motl_file.star', motl_type='stopgap').df) # the labels corresponds to the Motl object; to display the values we need to inspect the df attribute

#### Reading IMOD .mod files
Amongst the child classes of `Motl`, `ModMotl` allows to read binary .mod files from IMOD and work with them as ModMotl objects, which inherit the same parameters as Motl objects. Examples:

* Load one or multiple .mod files from IMOD and store them as ModMotl:

In [ ]:
my_modmotl = ModMotl('path/to/model_folder/', mod_prefix='tomo_') #loaad all mod files in model_folder that start with tomo_
my_modmotl = ModMotl('path/to/model_file.mod') #load model_file.mod

display(my_modmotl.df) # inspect the content

* Get the content of a model file as a pandas DataFrame object:

In [ ]:
mod_df = ModMotl.read_in('path/to/model_file.mod') # read in a mod file and return a pandas dataframe

Conversely, to convert a Motl and write it out to a binary .mod file, the `write_to_model_file()` function is available:

In [ ]:
my_motl.write_to_model_file("tomo_id", "tomo", point_size=1) #split my_motl based on the tomo_id and write the resulting motl files to individual model files with the prefix "tomo_"

## CLI commands

As mentioned above, the user-facing operations for this module (as for the other core modules) from the command line interface. This allows to automate analysis pipelines in bash scripts. <br>
To get a complete list of the available commands type:

In [ ]:
cc_cryomotl --help

To get the details about all the arguments for a specific command, type the following:

In [ ]:
cc_cryomotl <subcommand> --help

Note that the CLI commands require the path of an output file as mandatory. This is specified by the `--output_motl` argument. An example is provided below for scaling coordinates:

In [ ]:
cc_cryomotl scale_coordinates  --input_motl "au_1.em" --output_motl "path/to/output/au_1_x2.em" --scaling_factor 2